# MPI Theory Verification: MoE with Zhang Invariant

This notebook verifies the **Mathematical Principles of Intelligence (MPI)** theory, specifically the **Zhang Invariant**, using a **Mixture of Experts (MoE)** architecture similar to MiniMind.

## Theory
Traditional MoE routers use a simple `Softmax(xW)` to select experts. MPI predicts that this leads to **Expert Collapse** (a few experts doing all the work) or **Signal Dissipation**.

We propose injecting the **Zhang Invariant** via **Sinkhorn Projection** (Birkhoff Polytope) into the routing mechanism. This theoretically guarantees:
1. **Load Balancing**: Equal utilization of experts (Cognitive Democracy).
2. **Signal Conservation**: Zero dissipation of information during routing.

Let's simulate this comparison.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Kaggle/Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed(42)

## 1. The Core Algorithm: Sinkhorn Projection (Zhang Invariant)
This function projects the routing matrix onto the Birkhoff Polytope, ensuring it is doubly stochastic (rows sum to 1, columns sum to 1).

In [ ]:
def sinkhorn_projection(matrix, iterations=5):
    """
    Projects the router weights onto the Birkhoff Polytope (Doubly Stochastic).
    Ensures that the 'Energy' sent to experts and received by experts is conserved.
    """
    # Softmax first to ensure positivity
    M = torch.exp(matrix)
    
    for _ in range(iterations):
        # Row norm (Conservation of Token Energy)
        row_sum = M.sum(dim=1, keepdim=True) + 1e-6
        M = M / row_sum
        
        # Col norm (Conservation of Expert Capacity)
        col_sum = M.sum(dim=0, keepdim=True) + 1e-6
        M = M / col_sum
        
    return M

## 2. MoE Layer Implementation
We define a standard MoE layer. The only difference is the `mode` parameter.
* `mode='Baseline'`: Uses standard Softmax routing.
* `mode='MPI'`: Uses Sinkhorn routing (Zhang Invariant).

In [ ]:
class Expert(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim)
        self.w2 = nn.Linear(hidden_dim, dim)
        self.act = nn.SiLU()
        
    def forward(self, x):
        return self.w2(self.act(self.w1(x)))

class MoELayer(nn.Module):
    def __init__(self, dim, num_experts, k=2, mode='Baseline'):
        super().__init__()
        self.dim = dim
        self.num_experts = num_experts
        self.k = k
        self.mode = mode # 'Baseline' or 'MPI'
        
        # Router
        self.router = nn.Linear(dim, num_experts)
        
        # Experts
        self.experts = nn.ModuleList([Expert(dim, dim*4) for _ in range(num_experts)])
        
        self.temperature = 1.0

    def forward(self, x):
        batch_size = x.shape[0]
        
        # 1. Routing Logits
        router_logits = self.router(x) # [batch, num_experts]
        
        # --- MPI INTERVENTION ---
        if self.mode == 'MPI':
            # Step A: Normalize Logits
            router_logits = router_logits / self.temperature
            
            # Step B: Sinkhorn-like Balancing
            M = torch.exp(router_logits)
            for _ in range(3):
                M = M / (M.sum(dim=1, keepdim=True) + 1e-6) # Token constraint
                M = M / (M.sum(dim=0, keepdim=True) + 1e-6) # Expert constraint
            
            routing_weights = M
            
            # Select Top-K from the balanced matrix
            top_k_weights, top_k_indices = torch.topk(routing_weights, self.k, dim=1)
            
            # Re-normalize
            top_k_weights = top_k_weights / top_k_weights.sum(dim=1, keepdim=True)
            
        else: # Baseline (Standard Top-K)
            routing_weights = F.softmax(router_logits, dim=1)
            top_k_weights, top_k_indices = torch.topk(routing_weights, self.k, dim=1)
            top_k_weights = top_k_weights / top_k_weights.sum(dim=1, keepdim=True)

        # 2. Dispatch and Aggregate
        final_output = torch.zeros_like(x)
        
        # Simple Loop Dispatch
        for i in range(batch_size):
            for j in range(self.k):
                expert_idx = top_k_indices[i, j].item()
                weight = top_k_weights[i, j]
                expert_out = self.experts[expert_idx](x[i].unsqueeze(0))
                final_output[i] += weight * expert_out.squeeze(0)
                
        return final_output, top_k_indices

## 3. Running the Simulation
We generate synthetic data with a distribution shift and observe how the experts are utilized.

In [ ]:
# Settings
dim = 64
num_experts = 8
k = 2
num_tokens = 1000

# Data Generation (Cluster 1 and Cluster 2)
data1 = torch.randn(num_tokens // 2, dim) + torch.tensor([2.0] * dim)
data2 = torch.randn(num_tokens // 2, dim) - torch.tensor([2.0] * dim)
data = torch.cat([data1, data2], dim=0).to(device)

# Initialize Models
model_baseline = MoELayer(dim, num_experts, k=k, mode='Baseline').to(device)
model_mpi = MoELayer(dim, num_experts, k=k, mode='MPI').to(device)

# Ensure identical start
model_mpi.load_state_dict(model_baseline.state_dict())

# Run Inference
print("Running Baseline...")
_, indices_baseline = model_baseline(data)

print("Running MPI-Enhanced...")
_, indices_mpi = model_mpi(data)

# Flatten for analysis
indices_baseline = indices_baseline.cpu().flatten().numpy()
indices_mpi = indices_mpi.cpu().flatten().numpy()

## 4. Visualization & Analysis
We calculate the **Entropy** of expert utilization. 
* **Low Entropy**: Means only a few experts are used (Collapse).
* **High Entropy**: Means experts are used evenly (Ideal).

In [ ]:
def calculate_entropy(indices, n_experts):
    counts = np.bincount(indices, minlength=n_experts)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log(probs + 1e-10))

entropy_baseline = calculate_entropy(indices_baseline, num_experts)
entropy_mpi = calculate_entropy(indices_mpi, num_experts)

print(f"Baseline Expert Entropy: {entropy_baseline:.4f}")
print(f"MPI Expert Entropy:      {entropy_mpi:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hist(indices_baseline, bins=range(num_experts+1), rwidth=0.8, color='gray', alpha=0.7)
axes[0].set_title(f"Baseline MoE (Entropy: {entropy_baseline:.2f})\nRisk: Expert Collapse")
axes[0].set_xlabel("Expert ID")
axes[0].set_ylabel("Token Count")

axes[1].hist(indices_mpi, bins=range(num_experts+1), rwidth=0.8, color='green', alpha=0.7)
axes[1].set_title(f"MPI-Enhanced MoE (Entropy: {entropy_mpi:.2f})\nBenefit: Load Balancing (Zhang Invariant)")
axes[1].set_xlabel("Expert ID")

plt.tight_layout()
plt.show()